In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_7 (3).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_225x225_42 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_245x206_53 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_6 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_275x183_33.jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_15.jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_225x225_40 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_194x259_65.jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_15 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cat

In [2]:
import os

for dataset_root in os.listdir('/kaggle/input'):
    print(f"\n=== /kaggle/input/{dataset_root} ===")
    count = 0
    for dirname, _, filenames in os.walk(f'/kaggle/input/{dataset_root}'):
        for filename in filenames:
            print(os.path.join(dirname, filename))
            count += 1
            if count >= 5:
                break
        if count >= 5:
            break


=== /kaggle/input/datasets ===
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_7 (3).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_225x225_42 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_245x206_53 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_6 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_275x183_33.jpg

=== /kaggle/input/notebooks ===
/kaggle/input/notebooks/adityachute/cattle-breed-classifier/training_curves.png
/kaggle/input/notebooks/adityachute/cattle-breed-classifier/__results__.html
/kaggle/input/notebooks/adityachute/cattle-breed-classifier/__notebook__.ipynb
/kaggle/input/notebooks/adityachute/cattle-breed-classifier/__output__.json
/kaggle/input/notebooks/adityachute/cattle-breed-classifier/best_cattle_classifier_v2.pth


In [3]:
import os

for root in ['/kaggle/input/datasets/atharvadarpude', '/kaggle/input/datasets/lukex9442']:
    for dirname, _, filenames in os.walk(root):
        for filename in filenames[:3]:
            print(os.path.join(dirname, filename))
        if filenames:
            break

/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_7 (3).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_225x225_42 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_245x206_53 (2).jpg
/kaggle/input/datasets/lukex9442/indian-bovine-breeds/bovine_breeds_metadata.csv


In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset'):
    for filename in filenames[:3]:
        print(os.path.join(dirname, filename))
    if filenames:
        break

/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_259x194_7 (3).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_225x225_42 (2).jpg
/kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle/Purnea/image_245x206_53 (2).jpg


In [5]:
!cat /kaggle/input/datasets/raghavdharwal/cows-and-buffalo-computer-vision-dataset/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 12
names: ['0', '1', '10', '11', '2', '3', '4', '5', '6', '7', '8', '9']

roboflow:
  workspace: object-detection-naauz
  project: cow-buffalo
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/object-detection-naauz/cow-buffalo/dataset/1

In [6]:
%%writefile data_preparation.py
import argparse
import hashlib
import json
import os
import re
import shutil
from collections import defaultdict
from pathlib import Path

from PIL import Image
from tqdm import tqdm

try:
    import imagehash
    HAS_IMAGEHASH = True
except ImportError:
    HAS_IMAGEHASH = False
    print("[WARN] imagehash not installed -> near-duplicate removal will be skipped.")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
JUNK_FOLDER_NAMES = {"images", "image", "img", "data", "train", "test", "val", "dataset"}

BREED_ALIASES = {
    "alambadi": "Alambadi",
    "amritmahal": "Amritmahal",
    "ayrshire": "Ayrshire_Cross",
    "banni": "Banni",
    "bargur": "Bargur",
    "bhadawari": "Bhadawari", "bhadwari": "Bhadawari",
    "brownswiss": "Brown_Swiss_Cross",
    "chhattisgarhi": "Chhattisgarhi",
    "dangi": "Dangi",
    "deoni": "Deoni",
    "gir": "Gir", "girindia": "Gir", "gircattle": "Gir",
    "guernsey": "Guernsey_Cross",
    "hallikar": "Hallikar",
    "hariana": "Hariana",
    "himachalipahari": "Himachali_Pahari",
    "holsteinfriesian": "HF_Cross", "hf": "HF_Cross", "hfcross": "HF_Cross",
    "jaffarabadi": "Jaffarabadi", "jaffrabadi": "Jaffarabadi",
    "jersey": "Jersey_Cross", "jerseycross": "Jersey_Cross",
    "kalahandi": "Kalahandi",
    "kangayam": "Kangayam",
    "kankrej": "Kankrej",
    "kasargod": "Kasargod",
    "kenkatha": "Kenkatha",
    "khariar": "Khariar",
    "kherigarh": "Kherigarh",
    "khillari": "Khillari",
    "konkankapila": "Konkan_Kapila",
    "kosali": "Kosali",
    "krishnavalley": "Krishna_Valley",
    "ladakhi": "Ladakhi",
    "lakhimi": "Lakhimi",
    "malnadgidda": "Malnad_Gidda",
    "malvi": "Malvi",
    "mehsana": "Mehsana",
    "mewati": "Mewati",
    "murrah": "Murrah",
    "nagori": "Nagori",
    "nagpuri": "Nagpuri",
    "nari": "Nari",
    "nilravi": "Nili_Ravi", "niliravi": "Nili_Ravi",
    "nimari": "Nimari",
    "ongole": "Ongole",
    "pandharpuri": "Pandharpuri",
    "podathirupu": "Poda_Thirupu",
    "pulikulam": "Pulikulam",
    "punganur": "Punganur",
    "purnea": "Purnea",
    "rathi": "Rathi",
    "reddane": "Red_Dane_Cross",
    "redsindhi": "Red_Sindhi", "sindhired": "Red_Sindhi",
    "redkandhari": "Red_Kandhari",
    "sahiwal": "Sahiwal",
    "shwetakapila": "Shweta_Kapila",
    "surti": "Surti",
    "tharparkar": "Tharparkar",
    "toda": "Toda",
    "umblachery": "Umblachery",
    "vechur": "Vechur",
    "bachaur": "Bachaur",
    "badri": "Badri",
    "bhelai": "Bhelai",
    "chilika": "Chilika",
    "dagri": "Dagri",
    "gangatari": "Gangatiri",
    "gaolao": "Gaolao",
    "ghumsari": "Ghumsari",
    "gojri": "Gojri",
    "luit": "Luit",
    "marathwada": "Marathwadi",
    "motu": "Motu",
    "ponwar": "Ponwar",
    "siri": "Siri",
    "thutho": "Thutho",
    "crossbreed": "Crossbred_Cattle", "crossbred": "Crossbred_Cattle",
    "desicow": "Desi_Cattle", "desi": "Desi_Cattle",
    "desibuffalo": "Desi_Buffalo", "desibuffalos": "Desi_Buffalo",
}


def normalize_key(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower())


def canonical_breed_name(raw_folder_name: str) -> str:
    key = normalize_key(raw_folder_name)
    if key in BREED_ALIASES:
        return BREED_ALIASES[key]
    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", raw_folder_name).strip("_")
    return cleaned if cleaned else "Unknown"


def discover_breed_folders(raw_dir: Path):
    """If the folder containing images has a generic name (images/img/data),
    use its PARENT folder's name as the breed instead - some datasets nest
    images one level deeper as <Breed>/images/*.jpg."""
    found = []
    for root, dirs, files in os.walk(raw_dir):
        has_image = any(Path(f).suffix.lower() in IMAGE_EXTS for f in files)
        if not has_image:
            continue
        root_path = Path(root)
        if root_path.name.lower() in JUNK_FOLDER_NAMES:
            parent = root_path.parent
            if parent == raw_dir or parent.name.lower() in JUNK_FOLDER_NAMES:
                continue
            found.append((root_path, parent.name))
        else:
            found.append((root_path, root_path.name))
    return found


def is_valid_image(path: Path, min_side=64) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            w, h = img.size
            if w < min_side or h < min_side:
                return False
        return True
    except Exception:
        return False


def phash_of(path: Path):
    try:
        with Image.open(path) as img:
            return str(imagehash.phash(img.convert("RGB")))
    except Exception:
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw_dirs", nargs="+", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--min_images", type=int, default=15)
    ap.add_argument("--val_split", type=float, default=0.15)
    ap.add_argument("--test_split", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    breed_to_images = defaultdict(list)
    all_raw_names = set()

    for raw_dir in args.raw_dirs:
        raw_dir = Path(raw_dir)
        if not raw_dir.exists():
            print(f"[WARN] raw dir not found, skipping: {raw_dir}")
            continue
        folders = discover_breed_folders(raw_dir)
        print(f"[{raw_dir.name}] discovered {len(folders)} candidate breed folders")
        for folder_path, raw_name in folders:
            all_raw_names.add(raw_name)
            breed = canonical_breed_name(raw_name)
            imgs = [p for p in folder_path.iterdir()
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
            breed_to_images[breed].extend(imgs)

    print("\n=== All raw folder names discovered ===")
    for n in sorted(all_raw_names):
        print(" -", n)
    print("=" * 70)

    print("\nValidating images...")
    for breed in list(breed_to_images.keys()):
        valid = [p for p in tqdm(breed_to_images[breed], desc=breed, leave=False)
                 if is_valid_image(p)]
        breed_to_images[breed] = valid

    if HAS_IMAGEHASH:
        print("\nRemoving near-duplicate images...")
        seen_hashes = set()
        for breed in list(breed_to_images.keys()):
            deduped = []
            for p in tqdm(breed_to_images[breed], desc=breed, leave=False):
                h = phash_of(p)
                if h is None or h in seen_hashes:
                    continue
                seen_hashes.add(h)
                deduped.append(p)
            breed_to_images[breed] = deduped

    dropped = []
    for breed in list(breed_to_images.keys()):
        if len(breed_to_images[breed]) < args.min_images:
            dropped.append((breed, len(breed_to_images[breed])))
            del breed_to_images[breed]
    if dropped:
        print(f"\nDropped {len(dropped)} classes with < {args.min_images} images:")
        for b, c in dropped:
            print(f"   {b}: {c} images")

    import random
    random.seed(args.seed)

    report_rows = []
    for breed, imgs in sorted(breed_to_images.items()):
        random.shuffle(imgs)
        n = len(imgs)
        n_test = max(1, int(n * args.test_split))
        n_val = max(1, int(n * args.val_split))
        n_train = n - n_val - n_test

        splits = {
            "train": imgs[:n_train],
            "val": imgs[n_train:n_train + n_val],
            "test": imgs[n_train + n_val:],
        }

        for split_name, split_imgs in splits.items():
            split_dir = out_dir / split_name / breed
            split_dir.mkdir(parents=True, exist_ok=True)
            for src in split_imgs:
                h = hashlib.md5(str(src).encode()).hexdigest()[:8]
                dst = split_dir / f"{h}_{src.name}"
                shutil.copy2(src, dst)

        report_rows.append({
            "breed": breed, "total": n,
            "train": len(splits["train"]), "val": len(splits["val"]), "test": len(splits["test"]),
        })

    import csv
    report_path = out_dir / "merge_report.csv"
    with open(report_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["breed", "total", "train", "val", "test"])
        writer.writeheader()
        writer.writerows(report_rows)

    class_names = sorted(breed_to_images.keys())
    with open(out_dir / "class_names.json", "w") as f:
        json.dump(class_names, f, indent=2)

    print(f"\nDone. {len(class_names)} breed classes, "
          f"{sum(r['total'] for r in report_rows)} images total.")
    print(f"Report: {report_path}")


if __name__ == "__main__":
    main()

Writing data_preparation.py


In [7]:
!head -5 data_preparation.py

import argparse
import hashlib
import json
import os
import re


In [8]:
!python data_preparation.py \
    --raw_dirs /kaggle/input/datasets/lukex9442/indian-bovine-breeds/Indian_bovine_breeds/Indian_bovine_breeds \
               /kaggle/input/datasets/atharvadarpude/indian-cattle-image-dataset/cattle \
               /kaggle/input/datasets/atharvadarpude/indian-buffalo-dataset/buffalo \
    --out_dir /kaggle/working/data/processed \
    --min_images 15

[Indian_bovine_breeds] discovered 41 candidate breed folders
[cattle] discovered 50 candidate breed folders
[buffalo] discovered 17 candidate breed folders

=== All raw folder names discovered ===
 - Alambadi
 - Amritmahal
 - Ayrshire
 - Banni
 - Bargur
 - Bhadawari
 - Brown_Swiss
 - Chhattisgarhi
 - Dangi
 - Deoni
 - Gir
 - Guernsey
 - Hallikar
 - Hariana
 - Himachali Pahari
 - Holstein_Friesian
 - Jaffarabadi
 - Jaffrabadi
 - Jersey
 - Kangayam
 - Kankrej
 - Kasargod
 - Kenkatha
 - Khariar
 - Kherigarh
 - Khillari
 - Konkan Kapila
 - Kosali
 - Krishna_Valley
 - Ladakhi
 - Lakhimi
 - Malnad_gidda
 - Mehsana
 - Mewati
 - Murrah
 - Nagori
 - Nagpuri
 - Nari
 - Nili_Ravi
 - Nimari
 - Ongole
 - Poda Thirupu
 - Pulikulam
 - Punganur
 - Purnea
 - Rathi
 - Red kandhari
 - Red_Dane
 - Red_Sindhi
 - Sahiwal
 - Shweta Kapila
 - Surti
 - Tharparkar
 - Toda
 - Umblachery
 - Vechur
 - bachaur
 - badri
 - banni
 - bargur
 - bhadwari
 - bhelai
 - chilika
 - dagri
 - gangatari
 - gaolao
 - ghumsari
 

In [9]:
!cat /kaggle/working/data/processed/merge_report.csv

In [10]:
!nvidia-smi

Fri Jul 10 09:00:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
%%writefile train.py
"""
Phase 2 - Model Training
"""

import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight


def build_datasets(data_dir: Path, img_size: int, batch_size: int):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "train", image_size=(img_size, img_size),
        batch_size=batch_size, label_mode="int", shuffle=True, seed=42,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "val", image_size=(img_size, img_size),
        batch_size=batch_size, label_mode="int", shuffle=False,
    )
    class_names = train_ds.class_names

    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomContrast(0.15),
        tf.keras.layers.RandomBrightness(0.15),
    ], name="augmentation")

    preprocess = tf.keras.applications.mobilenet_v3.preprocess_input

    def prep_train(x, y):
        x = augment(x, training=True)
        x = preprocess(x)
        return x, y

    def prep_eval(x, y):
        x = preprocess(x)
        return x, y

    train_ds_prepped = train_ds.map(prep_train, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds_prepped = val_ds.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE)

    train_ds_prepped = train_ds_prepped.prefetch(tf.data.AUTOTUNE)
    val_ds_prepped = val_ds_prepped.prefetch(tf.data.AUTOTUNE)

    return train_ds, train_ds_prepped, val_ds_prepped, class_names


def compute_weights(train_ds, num_classes):
    labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
    classes = np.arange(num_classes)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def build_model(num_classes: int, img_size: int):
    base = tf.keras.applications.MobileNetV3Large(
        input_shape=(img_size, img_size, 3), include_top=False, weights="imagenet",
        pooling="avg",
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--img_size", type=int, default=224)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--epochs_head", type=int, default=15)
    ap.add_argument("--epochs_finetune", type=int, default=20)
    ap.add_argument("--unfreeze_layers", type=int, default=40)
    ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--lr_finetune", type=float, default=5e-5)
    args = ap.parse_args()

    data_dir = Path(args.data_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_ds_raw, train_ds, val_ds, class_names = build_datasets(
        data_dir, args.img_size, args.batch_size)
    num_classes = len(class_names)
    print(f"Classes ({num_classes}): {class_names}")

    with open(out_dir / "class_names.json", "w") as f:
        json.dump(class_names, f, indent=2)

    class_weights = compute_weights(train_ds_raw, num_classes)
    print("Class weights:", class_weights)

    model, base = build_model(num_classes, args.img_size)

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "best_head.keras"), save_best_only=True, monitor="val_accuracy"),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ]

    print("\n=== Phase A: training classification head (backbone frozen) ===")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(args.lr_head),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    history_a = model.fit(
        train_ds, validation_data=val_ds,
        epochs=args.epochs_head, class_weight=class_weights, callbacks=callbacks,
    )

    print(f"\n=== Phase B: fine-tuning top {args.unfreeze_layers} backbone layers ===")
    base.trainable = True
    for layer in base.layers[:-args.unfreeze_layers]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(args.lr_finetune),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks_ft = [
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "best_finetuned.keras"), save_best_only=True, monitor="val_accuracy"),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-8),
    ]
    history_b = model.fit(
        train_ds, validation_data=val_ds,
        epochs=args.epochs_finetune, class_weight=class_weights, callbacks=callbacks_ft,
    )

    final_path = out_dir / "breed_model_final.keras"
    model.save(final_path)
    print(f"\nSaved final model to {final_path}")

    hist = {
        "phase_a": {k: [float(v) for v in vals] for k, vals in history_a.history.items()},
        "phase_b": {k: [float(v) for v in vals] for k, vals in history_b.history.items()},
    }
    with open(out_dir / "training_history.json", "w") as f:
        json.dump(hist, f, indent=2)


if __name__ == "__main__":
    main()

Writing train.py


In [12]:
!python train.py \
    --data_dir /kaggle/working/data/processed \
    --out_dir /kaggle/working/model \
    --epochs_head 15 --epochs_finetune 20 --img_size 224 --batch_size 32

Found 8402 files belonging to 73 classes.
I0000 00:00:1783674051.935064      67 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783674051.938105      67 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Found 1750 files belonging to 73 classes.
Classes (73): ['Alambadi', 'Amritmahal', 'Ayrshire_Cross', 'Bachaur', 'Badri', 'Banni', 'Bargur', 'Bhadawari', 'Bhelai', 'Brown_Swiss_Cross', 'Chhattisgarhi', 'Chilika', 'Dagri', 'Dangi', 'Deoni', 'Gangatiri', 'Gaolao', 'Ghumsari', 'Gir', 'Gojri', 'Guernsey_Cross', 'HF_Cross', 'Hallikar', 'Hariana', 'Himachali_Pahari', 'Jaffarabadi', 'Jersey_Cross', 'Kalahandi', 'Kangayam', 'Kankrej', 'Kasargod', 'Kenkatha', 'Khariar', 'Kherigarh', 'Khillari', 'Konkan_Kapila', 'Kosali',

In [13]:
%%writefile evaluate.py
"""
Phase 3 - Evaluation
"""

import argparse
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--model_path", required=True)
    ap.add_argument("--img_size", type=int, default=224)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--out_dir", default=None)
    args = ap.parse_args()

    data_dir = Path(args.data_dir)
    model_path = Path(args.model_path)
    out_dir = Path(args.out_dir) if args.out_dir else model_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir / "test", image_size=(args.img_size, args.img_size),
        batch_size=args.batch_size, label_mode="int", shuffle=False,
    )
    class_names = test_ds.class_names

    preprocess = tf.keras.applications.mobilenet_v3.preprocess_input
    test_ds_prepped = test_ds.map(lambda x, y: (preprocess(x), y)).prefetch(tf.data.AUTOTUNE)

    print(f"Loading model from {model_path} ...")
    model = tf.keras.models.load_model(model_path)

    y_true, y_pred_probs = [], []
    for x_batch, y_batch in test_ds_prepped:
        preds = model.predict(x_batch, verbose=0)
        y_pred_probs.append(preds)
        y_true.append(y_batch.numpy())

    y_true = np.concatenate(y_true)
    y_pred_probs = np.concatenate(y_pred_probs)
    y_pred = np.argmax(y_pred_probs, axis=1)

    report = classification_report(
        y_true, y_pred, target_names=class_names, digits=3, zero_division=0)
    print("\n" + report)
    with open(out_dir / "classification_report.txt", "w") as f:
        f.write(report)

    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, digits=3,
        zero_division=0, output_dict=True)
    with open(out_dir / "classification_report.json", "w") as f:
        json.dump(report_dict, f, indent=2)

    top3_correct = 0
    for i, probs in enumerate(y_pred_probs):
        top3 = np.argsort(probs)[-3:]
        if y_true[i] in top3:
            top3_correct += 1
    top3_acc = top3_correct / len(y_true)
    print(f"\nTop-3 Accuracy: {top3_acc:.4f}")
    with open(out_dir / "top3_accuracy.txt", "w") as f:
        f.write(f"Top-3 Accuracy: {top3_acc:.4f}\n")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(8, len(class_names) * 0.5), max(6, len(class_names) * 0.5)))
    sns.heatmap(cm, annot=False, cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted breed")
    plt.ylabel("True breed")
    plt.title("Confusion Matrix - Breed Classification")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = out_dir / "confusion_matrix.png"
    plt.savefig(cm_path, dpi=150)
    print(f"Confusion matrix saved to {cm_path}")

    print(f"\nAll evaluation artifacts saved in: {out_dir}")


if __name__ == "__main__":
    main()

Writing evaluate.py


In [14]:
!python evaluate.py \
    --data_dir /kaggle/working/data/processed \
    --model_path /kaggle/working/model/breed_model_final.keras

Found 1749 files belonging to 73 classes.
I0000 00:00:1783676457.450315    1194 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783676457.455930    1194 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Loading model from /kaggle/working/model/breed_model_final.keras ...
2026-07-10 09:41:09.593681: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-10 09:41:09.732851: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investig

In [15]:
!ls /kaggle/working/model/

best_finetuned.keras	 classification_report.json  confusion_matrix.png
best_head.keras		 classification_report.txt   top3_accuracy.txt
breed_model_final.keras  class_names.json	     training_history.json


In [16]:
from IPython.display import FileLink
FileLink('/kaggle/working/model/breed_model_final.keras')

/kaggle/working/model/breed_model_final.keras

In [17]:
!python evaluate.py \
    --data_dir /kaggle/working/data/processed \
    --model_path /kaggle/working/model/breed_model_final.keras

Found 1749 files belonging to 73 classes.
I0000 00:00:1783676508.055854    1954 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783676508.058248    1954 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Loading model from /kaggle/working/model/breed_model_final.keras ...
2026-07-10 09:42:00.379683: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-10 09:42:00.517578: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investig

In [18]:
!ls -la /kaggle/working/model/

total 77808
drwxr-xr-x 2 root root     4096 Jul 10 09:41 .
drwxr-xr-x 4 root root     4096 Jul 10 09:40 ..
-rw-r--r-- 1 root root 31656438 Jul 10 09:38 best_finetuned.keras
-rw-r--r-- 1 root root 15859095 Jul 10 09:16 best_head.keras
-rw-r--r-- 1 root root 31656438 Jul 10 09:40 breed_model_final.keras
-rw-r--r-- 1 root root    10480 Jul 10 09:42 classification_report.json
-rw-r--r-- 1 root root     4545 Jul 10 09:42 classification_report.txt
-rw-r--r-- 1 root root     1030 Jul 10 09:00 class_names.json
-rw-r--r-- 1 root root   451638 Jul 10 09:42 confusion_matrix.png
-rw-r--r-- 1 root root       23 Jul 10 09:42 top3_accuracy.txt
-rw-r--r-- 1 root root     4926 Jul 10 09:40 training_history.json
